# Data Exploration

## Transform `txt` format data into `csv` format data.

http://www.cs.cmu.edu/~dbamman/booksummaries.html

In [17]:
import os
import sys
import pandas as pd

TXT_FILE = "../data/raw/booksummaries.txt"
CSV_FILE = "../data/raw/booksummaries.csv"

if not os.path.exists(TXT_FILE):
    sys.exit(f"File not found: {TXT_FILE}")

df = pd.read_csv(
    TXT_FILE,
    sep="\t",
    header=None,
    names=[
        "wikipedia_id",
        "freebase_id",
        "title",
        "author",
        "publication_date",
        "genres",
        "synopsis"
    ],
    quoting=3,          # QUOTE_NONE – important because of quotes inside summaries
    encoding="utf-8",
    on_bad_lines="skip" # in case there are a few broken lines
)

print(f"Loaded {len(df)} books")
print(df.info(), '\n')

df.to_csv(CSV_FILE, index=False, encoding="utf-8")

df.head(3)

Loaded 16559 books
<class 'pandas.DataFrame'>
RangeIndex: 16559 entries, 0 to 16558
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   wikipedia_id      16559 non-null  int64
 1   freebase_id       16559 non-null  str  
 2   title             16559 non-null  str  
 3   author            14177 non-null  str  
 4   publication_date  10949 non-null  str  
 5   genres            12841 non-null  str  
 6   synopsis          16559 non-null  str  
dtypes: int64(1), str(6)
memory usage: 42.1 MB
None 



,wikipedia_id,freebase_id,title,author,publication_date,genres,synopsis
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."
2,986,/m/0ldx,The Plague,Albert Camus,1947,"{""/m/02m4t"": ""Existentialism"", ""/m/02xlf"": ""Fi...",The text of The Plague is divided into five p...


## Imports

In [18]:
# Dataset management
import numpy as np
import pandas as pd
import os
import json

# NLP / Preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources (run once)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

## Load dataset

In [19]:
CSV_FILE = "../data/raw/booksummaries.csv"

if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
else:
    raise FileNotFoundError(f"Dataset not found at {CSV_FILE}")

In [20]:
df.head()

,wikipedia_id,freebase_id,title,author,publication_date,genres,synopsis
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."
2,986,/m/0ldx,The Plague,Albert Camus,1947,"{""/m/02m4t"": ""Existentialism"", ""/m/02xlf"": ""Fi...",The text of The Plague is divided into five p...
3,1756,/m/0sww,An Enquiry Concerning Human Understanding,David Hume,NaN,NaN,The argument of the Enquiry proceeds by a ser...
4,2080,/m/0wkt,A Fire Upon the Deep,Vernor Vinge,NaN,"{""/m/03lrw"": ""Hard science fiction"", ""/m/06n90...",The novel posits that space around the Milky ...


## Preprocessing

In [ ]:
# Change genres column into clear list
def parse_genre(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return np.nan
    try:
        text = json.loads(text)
        text = sorted(text.values())
        return ", ".join(text)
    except (json.JSONDecodeError, TypeError, AttributeError):
        return np.nan

df["genres"] = df["genres"].apply(parse_genre)

In [24]:
# Preprocess synopsis
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    - lower-case
    - tokenize with NLTK word_tokenize
    - keep only pure alphabetic tokens (letters only)
    - lemmatize (noun + verb) to reduce inflectional variants
    - drop stop-words that carry little topical signal
    """
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    tokens = word_tokenize(text)
    
    # Letters only, removes punctuation, numbers, mixed tokens
    tokens = [t for t in tokens if t.isalpha()]
    
    # Lemmatization (nouns then verbs)
    tokens = [lemmatizer.lemmatize(t, pos='n') for t in tokens]
    tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]
    
    # Remove stop words
    tokens = [t for t in tokens if t not in stop_words]
    
    return ' '.join(tokens)

# Apply preprocessing
df['synopsis'] = df['synopsis'].apply(preprocess_text)

In [25]:
df.head(5)

,wikipedia_id,freebase_id,title,author,publication_date,genres,synopsis
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"Children's literature, Fiction, Roman à clef, ...",old major old boar manor farm call animal farm...
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"Fiction, Novella, Satire, Science Fiction, Spe...",alex teenager live england lead gang nightly o...
2,986,/m/0ldx,The Plague,Albert Camus,1947,"Absurdist fiction, Existentialism, Fiction, Novel",text plague divide five part town oran thousan...
3,1756,/m/0sww,An Enquiry Concerning Human Understanding,David Hume,NaN,NaN,argument enquiry proceed series incremental st...
4,2080,/m/0wkt,A Fire Upon the Deep,Vernor Vinge,NaN,"Fantasy, Fiction, Hard science fiction, Scienc...",novel posit space around milky way divide conc...
